#### Importing Libraries

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from pathlib import Path

from sklearn.base import clone
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import ParameterGrid

#### Loading Feature-Engineered Datasets

In [2]:
data_dir = Path("../data/processed")
artifacts_dir = Path("../artifacts")
models_dir = Path("../models")

models_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(
    data_dir / "forecast_features_train.parquet"
)

test_df = pd.read_parquet(
    data_dir / "forecast_features_test.parquet"
)

with open(
    artifacts_dir / "forecast_feature_config.json",
    "r",
    encoding="utf-8"
) as file:
    feature_config = json.load(file)

forecast_horizon = feature_config["forecast_horizon"]

horizon_feature_columns = {
    int(horizon): columns
    for horizon, columns in feature_config["horizon_features"].items()
}

horizon_target_columns = {
    int(horizon): target
    for horizon, target in feature_config["horizon_targets"].items()
}

training_end_date = pd.Timestamp(
    feature_config["training_end_date"]
)

assert train_df["Target_Date_H7"].le(training_end_date).all()
assert test_df["Target_Date_H1"].gt(training_end_date).all()

assert set(horizon_feature_columns) == set(range(1, forecast_horizon + 1))
assert set(horizon_target_columns) == set(range(1, forecast_horizon + 1))

print("Training Dataset Shape:", train_df.shape)
print("Testing Dataset Shape:", test_df.shape)
print("Forecast Horizon:", forecast_horizon)
print("Training Cutoff:", training_end_date.date())

print("\nForecasting Dataset Summary:")
display(
    pd.DataFrame({
        "Dataset": ["Training", "Testing"],
        "Observations": [len(train_df), len(test_df)],
        "First_Origin": [
            train_df["Forecast_Origin"].min(),
            test_df["Forecast_Origin"].min()
        ],
        "Last_Origin": [
            train_df["Forecast_Origin"].max(),
            test_df["Forecast_Origin"].max()
        ]
    })
)

print("\nHorizon-Specific Predictor Counts:")
display(
    pd.DataFrame({
        "Horizon": list(horizon_feature_columns),
        "Predictors": [
            len(columns)
            for columns in horizon_feature_columns.values()
        ],
        "Target": [
            horizon_target_columns[horizon]
            for horizon in horizon_feature_columns
        ]
    })
)

Training Dataset Shape: (207, 87)
Testing Dataset Shape: (55, 87)
Forecast Horizon: 7
Training Cutoff: 2011-09-29

Forecasting Dataset Summary:


,Dataset,Observations,First_Origin,Last_Origin
0,Training,207,2011-01-16,2011-09-21
1,Testing,55,2011-09-29,2011-12-01



Horizon-Specific Predictor Counts:


,Horizon,Predictors,Target
0,1,48,Target_H1
1,2,48,Target_H2
2,3,48,Target_H3
3,4,48,Target_H4
4,5,48,Target_H5
5,6,48,Target_H6
6,7,48,Target_H7


**Key Observations**
- **Modeling Datasets:** The training dataset contains 207 forecasting observations, while the testing dataset contains 55, with 87 columns each and a chronological training cutoff of September 29, 2011.
- **Forecasting Configuration:** All seven forecast horizons have been loaded successfully, each containing 48 predictors and a corresponding revenue target.

#### Configuring Expanding-Window Validation

In [3]:
n_splits = 3
validation_size = 14

validation_folds = []

n_observations = len(train_df)
first_validation_start = n_observations - n_splits * validation_size

for fold in range(n_splits):
    validation_start = first_validation_start + fold * validation_size
    validation_end = validation_start + validation_size

    validation_indices = np.arange(
        validation_start,
        validation_end
    )

    first_validation_origin = train_df.loc[
        validation_start, "Forecast_Origin"
    ]

    eligible_training_indices = np.flatnonzero(
        train_df["Target_Date_H7"].le(first_validation_origin).to_numpy()
    )

    assert len(eligible_training_indices) > 0
    assert eligible_training_indices.max() < validation_start

    assert (
        train_df.loc[
            eligible_training_indices, "Target_Date_H7"
        ].max() <= first_validation_origin
    )

    validation_folds.append(
        (eligible_training_indices, validation_indices)
    )

fold_summary = pd.DataFrame([
    {
        "Fold": fold_number,
        "Training_Observations": len(training_indices),
        "Validation_Observations": len(validation_indices),
        "Training_Last_Origin": train_df.loc[
            training_indices[-1], "Forecast_Origin"
        ],
        "Training_Last_Target": train_df.loc[
            training_indices[-1], "Target_Date_H7"
        ],
        "Validation_First_Origin": train_df.loc[
            validation_indices[0], "Forecast_Origin"
        ],
        "Validation_Last_Origin": train_df.loc[
            validation_indices[-1], "Forecast_Origin"
        ]
    }
    for fold_number, (training_indices, validation_indices)
    in enumerate(validation_folds, start=1)
])

print("Expanding-Window Validation Configuration:")
print("Number of Folds:", len(validation_folds))
print("Validation Observations per Fold:", validation_size)

print("\nFold Summary:")
display(fold_summary)

print("\nValidation Checks:")
assert all(
    train_df.loc[training_indices, "Target_Date_H7"].max()
    <= train_df.loc[validation_indices[0], "Forecast_Origin"]
    for training_indices, validation_indices in validation_folds
)

print("All validation folds passed chronological leakage checks.")

Expanding-Window Validation Configuration:
Number of Folds: 3
Validation Observations per Fold: 14

Fold Summary:


,Fold,Training_Observations,Validation_Observations,Training_Last_Origin,Training_Last_Target,Validation_First_Origin,Validation_Last_Origin
0,1,159,14,2011-07-26,2011-08-03,2011-08-03,2011-08-18
1,2,173,14,2011-08-11,2011-08-19,2011-08-19,2011-09-05
2,3,187,14,2011-08-28,2011-09-06,2011-09-06,2011-09-21



Validation Checks:
All validation folds passed chronological leakage checks.


**Key Observations**
- **Expanding-Window Validation:** Three validation folds were established, each containing 14 forecast origins, with training observations increasing from 159 to 187 across successive folds.
- **Leakage Prevention:** All folds passed chronological validation checks, ensuring that training targets do not extend beyond the first validation forecast origin.

#### Establishing Baseline Forecasting Models

In [5]:
baseline_predictions = []

seasonal_period = 6
revenue_history = pd.concat(
    [
        train_df[["Forecast_Origin", "Origin_Revenue"]],
        test_df[["Forecast_Origin", "Origin_Revenue"]]
    ],
    ignore_index=True
).drop_duplicates(subset="Forecast_Origin")

revenue_history = (
    revenue_history
    .sort_values("Forecast_Origin")
    .reset_index(drop=True)
)

for fold_number, (training_indices, validation_indices) in enumerate(
    validation_folds,
    start=1
):
    validation_data = train_df.iloc[validation_indices]

    for horizon in range(1, forecast_horizon + 1):
        actual_values = validation_data[
            horizon_target_columns[horizon]
        ].to_numpy()

        naive_predictions = validation_data[
            "Origin_Revenue"
        ].to_numpy()

        seasonal_lag = (seasonal_period - horizon % seasonal_period) % seasonal_period

        if seasonal_lag == 0:
            seasonal_predictions = validation_data[
                "Origin_Revenue"
            ].to_numpy()
        else:
            seasonal_predictions = revenue_history[
                "Origin_Revenue"
            ].shift(seasonal_lag).reindex(
                validation_data.index
            ).to_numpy()

        for model_name, predictions in [
            ("Naive", naive_predictions),
            ("Seasonal_Naive", seasonal_predictions)
        ]:
            baseline_predictions.append({
                "Fold": fold_number,
                "Horizon": horizon,
                "Model": model_name,
                "MAE": mean_absolute_error(
                    actual_values, predictions
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        actual_values, predictions
                    )
                )
            })

baseline_results_df = pd.DataFrame(baseline_predictions)

baseline_summary_df = (
    baseline_results_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

print("Baseline Forecasting Performance:")
display(baseline_summary_df.round(2))

print("\nBaseline Performance by Horizon:")
display(
    baseline_results_df
    .groupby(["Model", "Horizon"], as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean")
    )
    .round(2)
)

print("\nBaseline Evaluation Completed Successfully.")

Baseline Forecasting Performance:


,Model,Mean_MAE,Mean_RMSE
0,Seasonal_Naive,13194.90,17086.83
1,Naive,18658.22,24089.94



Baseline Performance by Horizon:


,Model,Horizon,Mean_MAE,Mean_RMSE
0,Naive,1,20029.21,25075.50
1,Naive,2,18880.23,25936.44
2,Naive,3,20767.65,26856.53
3,Naive,4,19856.72,25159.75
4,Naive,5,17970.71,23556.93
5,Naive,6,14218.33,18407.34
6,Naive,7,18884.66,23637.07
7,Seasonal_Naive,1,12259.32,15899.58
8,Seasonal_Naive,2,12401.03,16054.24
9,Seasonal_Naive,3,12696.48,16232.43



Baseline Evaluation Completed Successfully.


**Key Observations**
- **Baseline Performance:** Seasonal Naive achieved the lowest average forecasting errors, with an MAE of 13,194.90 and RMSE of 17,086.83, outperforming the Naive baseline (MAE: 18,658.22; RMSE: 24,089.94).
- **Seasonal Predictive Value:** Seasonal Naive reduced average MAE by approximately 29.3% compared with Naive forecasting, indicating that historical six-trading-day revenue patterns provide a useful benchmark for subsequent machine learning models.

#### Visualizing Baseline Forecasting Performance

In [6]:
import plotly.express as px

baseline_comparison_df = (
    baseline_results_df
    .groupby(["Model", "Horizon"], as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean")
    )
)

fig = px.line(
    baseline_comparison_df,
    x="Horizon",
    y="Mean_MAE",
    color="Model",
    markers=True,
    title="Baseline Forecasting Error Across Prediction Horizons",
    labels={
        "Horizon": "Forecast Horizon (Observed Trading Days)",
        "Mean_MAE": "Mean Absolute Error",
        "Model": "Baseline Model"
    },
    template="plotly_white"
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    legend_title_text="Baseline Model"
)

fig.update_xaxes(
    dtick=1
)

fig.show()

print("Baseline Forecasting Performance by Horizon:")
display(
    baseline_comparison_df
    .sort_values(["Model", "Horizon"])
    .round(2)
)

Baseline Forecasting Performance by Horizon:


,Model,Horizon,Mean_MAE,Mean_RMSE
0,Naive,1,20029.21,25075.50
1,Naive,2,18880.23,25936.44
2,Naive,3,20767.65,26856.53
3,Naive,4,19856.72,25159.75
4,Naive,5,17970.71,23556.93
5,Naive,6,14218.33,18407.34
6,Naive,7,18884.66,23637.07
7,Seasonal_Naive,1,12259.32,15899.58
8,Seasonal_Naive,2,12401.03,16054.24
9,Seasonal_Naive,3,12696.48,16232.43


**Key Observations**
- **Consistent Seasonal Advantage:** Seasonal Naive achieves lower MAE and RMSE than the standard Naive baseline across all seven forecast horizons, demonstrating the predictive value of recurring six-trading-day revenue patterns.
- **Horizon-Wise Performance:** Seasonal Naive records its lowest MAE at Horizon 1 (12,259.32) and highest at Horizon 5 (14,210.94), indicating moderate variation in forecasting accuracy across the prediction horizon.